# AI-ML Assignment – 3
## Salary Prediction using Polynomial Regression

**Dataset:** Position Salaries Dataset  
**Total Marks:** 10  
**Submission Deadline:** 23 July 2026, 11:59 PM IST

---
## Task 1: Data Understanding (2 Marks)

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# 1. Load the dataset using Pandas
df = pd.read_csv('Position_Salaries.csv')
print('Dataset loaded successfully!')
print(f'Shape: {df.shape}')

In [ ]:
# 2. Display the first five records
print('First 5 Records:')
df.head()

In [ ]:
# 3. Identify Input Feature and Target Variable
print('='*50)
print('Feature Identification')
print('='*50)
print(f'Input Feature  : Level  (numeric job position level from 1 to 10)')
print(f'Target Variable: Salary (annual salary in USD)')
print()
print('Note: The "Position" column is a descriptive label and is NOT')
print('      used as a model feature since "Level" encodes the same info numerically.')

In [ ]:
# 4. Dataset information and summary statistics
print('--- Dataset Info ---')
df.info()
print()
print('--- Summary Statistics ---')
df.describe()

---
## Task 2: Data Preprocessing (2 Marks)

In [ ]:
# Check for missing values
print('Missing Values per Column:')
print(df.isnull().sum())
print()
if df.isnull().sum().sum() == 0:
    print('No missing values found. Dataset is clean.')
else:
    print('Missing values detected! Handle before proceeding.')

In [ ]:
# Select appropriate feature(s) and target variable
X = df[['Level']]          # Input feature (2D array required by sklearn)
y = df['Salary']           # Target variable

print(f'Feature matrix X shape : {X.shape}')
print(f'Target vector  y shape : {y.shape}')
print()
print('Feature (X) sample:')
print(X.values.flatten())
print('Target (y) sample:')
print(y.values)

In [ ]:
# Split the dataset into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Total samples   : {len(X)}')
print(f'Training samples: {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)')
print(f'Testing  samples: {len(X_test)}  ({len(X_test)/len(X)*100:.0f}%)')
print()
print('Training set (Level values):', X_train.values.flatten())
print('Testing  set (Level values):', X_test.values.flatten())

---
## Task 3: Model Development (3 Marks)

In [ ]:
# 1. Transform the input feature using Polynomial Features (Degree = 3)
poly = PolynomialFeatures(degree=3)

X_train_poly = poly.fit_transform(X_train)   # fit on training, transform training
X_test_poly  = poly.transform(X_test)        # transform test using same fitted poly

print(f'Original training feature shape : {X_train.shape}')
print(f'Polynomial training feature shape: {X_train_poly.shape}')
print()
print('Polynomial feature names:', poly.get_feature_names_out(['Level']))

In [ ]:
# 2. Train a Polynomial Regression model
# Polynomial Regression = Linear Regression applied on polynomial-transformed features
poly_reg = LinearRegression()
poly_reg.fit(X_train_poly, y_train)

print('Model trained successfully!')
print(f'Intercept : {poly_reg.intercept_:.4f}')
print(f'Coefficients: {poly_reg.coef_}')

In [ ]:
# 3. Predict salaries for the test dataset
y_pred = poly_reg.predict(X_test_poly)

print('Salary Predictions on Test Set:')
print('='*45)
print(f'{"Level":<10} {"Actual Salary":>15} {"Predicted Salary":>18}')
print('-'*45)
for level, actual, predicted in zip(X_test.values.flatten(), y_test.values, y_pred):
    print(f'{int(level):<10} {actual:>15,.0f} {predicted:>18,.0f}')
print('='*45)

---
## Task 4: Model Evaluation (2 Marks)

In [ ]:
# Evaluate the model using MAE, MSE, and R² Score
mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print('Model Evaluation Metrics (on Test Set):')
print('='*45)
print(f'Mean Absolute Error  (MAE)  : {mae:>15,.2f}')
print(f'Mean Squared Error   (MSE)  : {mse:>15,.2f}')
print(f'Root Mean Sq. Error  (RMSE) : {rmse:>15,.2f}')
print(f'R² Score                    : {r2:>15.4f}')
print('='*45)
print()
# Also evaluate on full dataset for a comprehensive view
X_poly_all = poly.transform(X)
y_pred_all = poly_reg.predict(X_poly_all)

r2_train = r2_score(y, y_pred_all)
print(f'R² Score on Full Dataset    : {r2_train:.4f}')

In [ ]:
# Visualization: Scatter plot of original data + Polynomial Regression Curve
X_range = np.linspace(X['Level'].min(), X['Level'].max(), 300).reshape(-1, 1)
X_range_poly = poly.transform(X_range)
y_range_pred = poly_reg.predict(X_range_poly)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ---- Plot 1: Full dataset ----
axes[0].scatter(X, y, color='steelblue', s=100, zorder=5, label='Actual Data')
axes[0].plot(X_range, y_range_pred, color='crimson', linewidth=2.5, label='Poly Regression (deg=3)')
axes[0].set_title('Salary vs Position Level\n(Polynomial Regression – Degree 3)', fontsize=13)
axes[0].set_xlabel('Position Level', fontsize=11)
axes[0].set_ylabel('Salary (USD)', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# ---- Plot 2: Actual vs Predicted (test set) ----
axes[1].scatter(y_test, y_pred, color='darkorange', s=120, edgecolors='black', zorder=5)
min_val = min(y_test.min(), y_pred.min()) - 10000
max_val = max(y_test.max(), y_pred.max()) + 10000
axes[1].plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1.5, label='Perfect Prediction')
axes[1].set_title('Actual vs Predicted Salary\n(Test Set)', fontsize=13)
axes[1].set_xlabel('Actual Salary (USD)', fontsize=11)
axes[1].set_ylabel('Predicted Salary (USD)', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('polynomial_regression_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved as polynomial_regression_results.png')

### Observations

**Observation 1 – Strong Model Fit:**  
The R² Score on the full dataset is very high (close to 1.0), indicating that the degree-3 polynomial regression model captures the non-linear growth pattern in salaries extremely well across all position levels.

**Observation 2 – Salary Growth is Exponential, Not Linear:**  
From the scatter plot, it is clearly visible that salary does not increase at a constant rate — it accelerates sharply at higher position levels (e.g., from Partner to C-level to CEO). The polynomial curve fits this curvature naturally, which a simple straight line would fail to capture.

**Observation 3 – Small Test Set Limitation:**  
Since the dataset contains only 10 rows, the 80/20 split yields just 2 test samples. The MAE and MSE on the test set can therefore fluctuate significantly depending on which samples end up in the test split. Evaluation on the full dataset provides a more stable performance picture for datasets of this size.

---
## Task 5: Conclusion (1 Mark)

### Key Findings
The Polynomial Regression model (degree = 3) successfully learned the non-linear relationship between employee position level and salary in the Position Salaries dataset. The model achieved a very high R² score, confirming that position level alone is a strong predictor of salary when modelled with a polynomial function.

### Linear Regression vs Polynomial Regression
Linear Regression assumes a straight-line relationship between the input feature and the target, expressed as `y = b0 + b1*x`. Polynomial Regression extends this by introducing higher-order terms (x², x³, etc.), allowing the model to fit curved, non-linear patterns in the data. In essence, polynomial regression is still a linear model — but linear in the *coefficients*, not in the *features*.

### Advantage of Polynomial Regression for This Dataset
The salary data exhibits a steep, accelerating growth curve — salaries rise modestly at junior levels but jump dramatically at executive levels (e.g., from 200,000 to 1,000,000 between Partner and CEO). A linear model would significantly underestimate high-level salaries and overestimate mid-level ones. Polynomial Regression captures this curvature precisely, making it the right tool for this dataset.
